# Lesson 7 : Agent Skills

Agent Skill is introduced by Anthropic on October 2025 and now it's an open (and cross-platform) project. (See [here](https://agentskills.io/home) for Agent Skill's specification.)  
The objective of Agent Skills is to enable highly modularization and reuse of actions and knowledge - such as, instructions, tool usage, and knowledge retrieval. Unlike tools in agents, Agent Skills allow you to group many assets into one module, so even large job-level tasks can be defined as a single module. (And yet, only those names and descriptions are set in the list of skills, so it doesn't consume much context.)

In this exercise, we create a brief custom skill to create a expense report, and we then use this skill with Agent Framework.

## Create a file-based skill

There exist a lot of pre-built (reusable) existing skills (see [here](https://github.com/heilcheng/awesome-agent-skills)), but in this exercise, we briefly build our own custom file skill to create a unique expense report for our virtual company as follows.

First we create a file (named ```SKILL.md```) to describe skill's body.

In [1]:
import os
skill_folder = path = os.path.join("skills", "corporate-expense-report")
os.makedirs(skill_folder, exist_ok=True)

In [2]:
%%writefile skills/corporate-expense-report/SKILL.md
---
name: corporate-expense-report
description: This skill calculates additional fee in each expense item and create a corporate expense report using template.
---

# Corporate Expense Report Skill

This skill provides instructions on how to create a corporate expense report.

## Capabilities

- Infer the category from the description in each expense item.  
  Available categories: "international transport", "domestic transport", "meal", and "misc"
- Calculate additional fee (tax + transaction fee) for each item
- Generate a expense report including additional fee

## Input Format

Provide the list of expense - in which each item includes description, date, and amount.

## Output Format

Show expense report using template: [assets/expense_template.md](assets/expense_template.md)

## Additional Fee

Additional fee in each item consists of tax fee and transaction fee.  
All of these costs are calculated by multiplying the expenses by a certain multiplier, and the multipliers in each category are listed in table on [assets/additional_fees.md](assets/additional_fees.md).

In order to run exact calculation, use contoso-utilities skill for all arithmetic operations.

Writing skills/corporate-expense-report/SKILL.md


In above skill's body, we refer assets - ```expense_template.md``` and ```additional_fees.md``` - which include output template for expense report and a list of tax/transaction ratio, respectively.  
Now we create these assets as follows.

In [3]:
asset_folder = path = os.path.join(skill_folder, "assets")
os.makedirs(asset_folder, exist_ok=True)

In [4]:
%%writefile skills/corporate-expense-report/assets/expense_template.md
| Date | Category | Amount (USD) | Additional fee (USD) | Sub total (USD) |
|------|----------|--------------|----------------------|-----------------|
|      |          |              |                      |                 |

Total: [total fee]

Writing skills/corporate-expense-report/assets/expense_template.md


In [5]:
%%writefile skills/corporate-expense-report/assets/additional_fees.md
| Category                | Tax percentage | Transaction percentage |
|-------------------------|----------------|------------------------|
| international transport | 0 %            | 10 %                   |
| domestic transport      | 0 %            | 5 %                    |
| meal                    | 10 %           | 0 %                    |
| misc                    | 3 %            | 0 %                    |

Writing skills/corporate-expense-report/assets/additional_fees.md


The ```skills``` folder will then have the following structure.  
In this exercise, we use only a single file skill to solve the problem, but you can include a lot of existing skills in ```skills``` folder.

```
skills/
└── corporate-expense-report/
    ├── SKILL.md
    └── assets/
        ├── expense_template.md
        └── additional_fees.md
```

## Create a code-defined skill

Sometimes we need a resource to execute in-process logic code, rather than static text.  
In such cases, we have used in-process local function tools, MCP tool calling, or code interpreter tools in previous examples.  
But in Skill's framework in Agent Framework, we can use code-defined skills to tackle with these scenarios.

In this example, we define a code-defined tool that performs primitive arithmetic operations, so that even large numbers can be calculated correctly (without calculation errors).

> Note : It is also possible to create code (.py) for tasks (such as, conversion, creation, ...) and register them as skill's assets in file-based skills.  
> Unlike code-defined skills, these tasks are executed in sandbox execution context, not in-process. (Python packages for task execution will also be installed as needed.)

In [6]:
from agent_framework import Skill

contoso_utilities_skill = Skill(
    name="contoso-utilities",
    description="Some utlities for achieving Contoso business",
    content="Use this skill when additional utilities for Contoso company is required.",
)

@contoso_utilities_skill.script(
    name="addition",
    description="Add two numbers",
)
def add(a: int, b: int) -> int:
    """Add two numbers

    Args:
        a: left-side number
        b: right-side number

    Returns:
        The result of addtion
    """
    return a + b

@contoso_utilities_skill.script(
    name="subtraction",
    description="Subtract two numbers",
)
def sub(a: int, b: int) -> int:
    """Subtract two numbers

    Args:
        a: left-side number
        b: right-side number

    Returns:
        The result of subtraction
    """
    return a + b

@contoso_utilities_skill.script(
    name="multiplication",
    description="Multiply two numbers",
)
def mul(a: int, b: int) -> int:
    """Multiply two numbers

    Args:
        a: left-side number
        b: right-side number

    Returns:
        The result of multiplication
    """
    return a * b

@contoso_utilities_skill.script(
    name="division",
    description="Divide two numbers",
)
def div(a: int, b: int) -> float:
    """Divide two numbers

    Args:
        a: left-side number
        b: right-side number

    Returns:
        The result of division
    """
    return a / b

## Run agent with skills

Now let's build an agent to use above skills - a file-based skill ("expense" skill) and a code-defined skill ("utilities" skill).

First we initilize the client object as usual.

In [7]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Now we define a skill provider, which refers to above skill's folder (which includes a file-based "expense" skill) and a code-defined skill.

In [8]:
from agent_framework import SkillsProvider

skills_provider = SkillsProvider(
    skill_paths="skills",
    skills=[contoso_utilities_skill]
)

Now we create an agent with this skill's provider as follows.  
Same as Lesson 6, we set a provider in ```context_providers``` property in the agent.

In [9]:
from agent_framework import Agent

agent = Agent(
    name="AgentWithSkills",
    client=client,
    instructions="You are a helpful assistant that can write and execute Python code to solve problems.",
    context_providers=[skills_provider],
)

Now let's run the agent.  
In this skill, the following ration of tax and transaction fee should be added in each item, and the agent shows an expense report with a table which has columns - "data", "category", "amount", "additional fee", and "sub total".

| category | tax ratio | transaction fee ratio |
|----------|-----------|-----------------------|
| international transport | 0.00 | 0.10 |
| domestic transport | 0.00 | 0.05 |
| meal | 0.10 | 0.00 |
| misc | 0.03 | 0.00 |

In [10]:
from IPython.display import Markdown, display

prompt = """Return an expense report of the following.

- flight from JFK to SEA (round trip)
    - date : 2026/02/09
    - amount : 2800
- dinner
    - date : 2026/02/09
    - amount : 80
- lunch
    - date : 2026/02/10
    - amount : 37
- souvenirs
    - date : 2026/02/10
    - amount : 40
"""

result = await agent.run(prompt)
display(Markdown(result.text))

| Date | Category | Amount (USD) | Additional fee (USD) | Sub total (USD) |
|------|----------|--------------|----------------------|-----------------|
| 2026/02/09 | domestic transport | 2800 | 140 | 2940 |
| 2026/02/09 | meal | 80 | 8 | 88 |
| 2026/02/10 | meal | 37 | 4 | 41 |
| 2026/02/10 | misc | 40 | 1 | 41 |

Total: 3110

Let's see how the skill was parsed and processed.  
As you can see, all basic calculations are handled one by one using "utlities" code-defined skills.

In [11]:
for i, msg in enumerate(result.messages):
    print(f"********** message {i} **********")
    for c in msg.contents:
        if c.type == "function_call":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function name : {c.name}")
            print(f"function arguments : {c.arguments}")
        elif c.type == "function_result":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function result : {c.result}")
            print(f"exceptions : {c.exception}")
        elif c.type == "text":
            print(f"*** {c.type} ***")
            print(f"text : {c.text}")
        else:
            print(f"*** Other types : {c.type}***")

********** message 0 **********
*** function_call ***
call id : call_gzV3IEe0l9PkenTS9pI7ELnL
function name : load_skill
function arguments : {"skill_name":"corporate-expense-report"}
********** message 1 **********
*** function_result ***
call id : call_gzV3IEe0l9PkenTS9pI7ELnL
function result : ---
name: corporate-expense-report
description: This skill calculates additional fee in each expense item and create a corporate expense report using template.
---

# Corporate Expense Report Skill

This skill provides instructions on how to create a corporate expense report.

## Capabilities

- Infer the category from the description in each expense item.  
  Available categories: "international transport", "domestic transport", "meal", and "misc"
- Calculate additional fee (tax + transaction fee) for each item
- Generate a expense report including additional fee

## Input Format

Provide the list of expense - in which each item includes description, date, and amount.

## Output Format

Show 